# Evaluating Agentic AI for Ontology Curation

This notebook presents results from a systematic evaluation of AI coding agents
on biomedical ontology editing tasks. We compare agent **harnesses** (the runtime
that orchestrates tool use and skill discovery) while holding the underlying model constant.

In [ ]:
import os
while not os.path.exists('analysis/scores.tsv'):
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ai4c_scribe.analysis import load_scores, summary_by_agent, balanced_comparison, paired_test

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
pd.set_option('display.precision', 3)

df = load_scores()

## 1. Dataset Overview

### Ontologies

We selected four biomedical ontologies covering different biological domains,
editing formats (OBO vs OWL), and levels of agent-config maturity.

In [ ]:
ont_summary = df.groupby('ontology').agg(
    n_runs=('f1', 'count'),
    n_cases=('issue_number', 'nunique'),
    mean_f1=('f1', 'mean'),
).round(3)
ont_summary

### Test cases

Each test case is a historical issue–PR pair. Cases span different editing operations and difficulty levels.

In [ ]:
case_summary = df.groupby(['case_type', 'difficulty']).agg(
    n_runs=('f1', 'count'),
    n_cases=('case', 'nunique'),
    mean_f1=('f1', 'mean'),
).round(3)
case_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby('case_type')['case'].nunique().sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Cases by Task Type')
axes[0].set_xlabel('Number of unique cases')
df.groupby('difficulty')['case'].nunique().reindex(['simple','medium','hard']).plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title('Cases by Difficulty')
axes[1].set_xlabel('Number of unique cases')
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_dataset_breakdown.png', dpi=150)
plt.show()

### Agents

An **agent** is the full configuration tuple: (harness, model, config_tag).
We use short codes like `CX.g55.v9` meaning Codex / gpt-5.5 / config version 9.

| Code prefix | Harness | Skills discovery |
|------------|---------|------------------|
| CC | Claude Code | `.claude/skills/` (native Skill tool) |
| CX | Codex CLI | `.agents/skills/` (native progressive disclosure) |
| OC | OpenCode | `.agents/` + `.claude/` (reads both) |

In [ ]:
agent_summary = summary_by_agent(df)
agent_summary

---
## 2. Main Result: Harness Comparison

The central question: **does the harness affect performance when model, skills, and
test cases are held constant?**

We compare Codex and OpenCode on gpt-5.5 with identical configs and cases using a
paired design (same case run on both harnesses).

In [ ]:
# Paired comparison: Codex vs OpenCode on shared cases
codex_by_case = df[df['runtime'] == 'codex'].groupby('case')['f1'].mean()
opencode_by_case = df[df['runtime'] == 'opencode'].groupby('case')['f1'].mean()
shared = codex_by_case.index.intersection(opencode_by_case.index)

c = codex_by_case.loc[shared].values
o = opencode_by_case.loc[shared].values
n_pairs = len(c)
diff = o - c

t_stat, p_val = stats.ttest_rel(o, c)
rng = np.random.default_rng(42)
boot = np.array([diff[rng.integers(0, n_pairs, n_pairs)].mean() for _ in range(10000)])
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
cohens_d = diff.mean() / diff.std(ddof=1)

print(f'Paired comparison: Codex vs OpenCode (gpt-5.5)')
print(f'  n = {n_pairs} paired cases')
print(f'  Codex mean F1:    {c.mean():.3f}')
print(f'  OpenCode mean F1: {o.mean():.3f}')
print(f'  Mean difference:  {diff.mean():+.3f}')
print(f'  Paired t({n_pairs-1}) = {t_stat:.3f}, p = {p_val:.4f}')
print(f'  95% bootstrap CI: [{ci_low:.3f}, {ci_high:.3f}]')
print(f'  Cohen\'s d = {cohens_d:.3f}')
print(f'  Significant at α=0.05: {"YES" if p_val < 0.05 else "NO"}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ax = axes[0]
ax.scatter(c, o, alpha=0.5, s=40)
ax.plot([0, 1.05], [0, 1.05], 'k--', alpha=0.3)
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('Codex F1'); ax.set_ylabel('OpenCode F1')
ax.set_title(f'Paired F1 (n={n_pairs})\np={p_val:.3f}')

ax = axes[1]
ax.hist(diff, bins=20, alpha=0.7, edgecolor='black')
ax.axvline(0, color='red', linestyle='--', label='No diff')
ax.axvline(diff.mean(), color='blue', label=f'Mean={diff.mean():.3f}')
ax.axvspan(ci_low, ci_high, alpha=0.15, color='blue')
ax.set_xlabel('F1 diff (OpenCode − Codex)'); ax.set_ylabel('Count')
ax.set_title('Paired Differences'); ax.legend(fontsize=9)

ax = axes[2]
diffs_by_d = []
labels = []
for d in ['simple', 'medium', 'hard']:
    sub = df[df['difficulty'] == d]
    cs = sub[sub['runtime']=='codex'].groupby('case')['f1'].mean()
    os_ = sub[sub['runtime']=='opencode'].groupby('case')['f1'].mean()
    sh = cs.index.intersection(os_.index)
    if len(sh) >= 3:
        diffs_by_d.append((os_.loc[sh] - cs.loc[sh]).values)
        labels.append(f'{d}\n(n={len(sh)})')
ax.boxplot(diffs_by_d, labels=labels)
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('F1 diff'); ax.set_title('By Difficulty')

plt.tight_layout()
plt.savefig('analysis/notebooks/fig_harness_comparison.png', dpi=150)
plt.show()

**Finding**: OpenCode outperforms Codex by a small but statistically significant margin
(mean diff ≈ 0.05 F1, p < 0.05). The effect is concentrated in medium-difficulty tasks.
Simple tasks hit a ceiling (both do well); hard tasks hit a floor (both struggle).

Practical implication: for controlled model comparisons, the harness should be held
constant or reported. A ~5% F1 difference from harness choice could mask or exaggerate
model differences of similar magnitude.

---
## 3. Performance by Ontology

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df[df['f1']>0], x='ontology', y='f1', hue='runtime', ax=ax)
ax.set_title('F1 by Ontology and Runtime (non-zero runs only)')
ax.set_ylabel('Metadiff F1'); ax.set_xlabel('')
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_by_ontology.png', dpi=150)
plt.show()

GO scores substantially higher than other ontologies (mean F1 ≈ 0.81 vs 0.37–0.53).
This reflects the maturity of the GO agent config (8 specialized skills for obsoletion,
reaction chemistry, design patterns, etc.) rather than inherent task difficulty —
GO cases include hard tasks that still score well because the skills provide
step-by-step procedural guidance.

---
## 4. Performance by Difficulty

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
order = ['simple', 'medium', 'hard']
sns.boxplot(data=df[df['f1']>0], x='difficulty', y='f1', order=order, ax=ax)
sns.stripplot(data=df[df['f1']>0], x='difficulty', y='f1', order=order, color='black', alpha=0.3, size=4, ax=ax)
ax.set_title('F1 by Difficulty')
ax.set_ylabel('Metadiff F1')

# Add means
for i, d in enumerate(order):
    m = df[df['difficulty']==d]['f1'].mean()
    ax.plot(i, m, 'D', color='red', markersize=8, zorder=5)

plt.tight_layout()
plt.savefig('analysis/notebooks/fig_by_difficulty.png', dpi=150)
plt.show()

print('Kruskal-Wallis test:')
groups = [df[df['difficulty']==d]['f1'].dropna().values for d in order]
h, p = stats.kruskal(*groups)
print(f'  H={h:.2f}, p={p:.4f}')

---
## 5. Complete Results Table

In [ ]:
pivot = df.pivot_table(values='f1', index='case', columns='agent', aggfunc='mean').round(3)
pivot.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=1)

---
## 6. Summary Statistics

In [ ]:
print(f'Total scored runs: {len(df)}')
print(f'Unique cases: {df["case"].nunique()}')
print(f'Unique agents: {df["agent"].nunique()}')
print(f'Success rate (F1 > 0): {(df["f1"]>0).mean()*100:.0f}%')
print(f'Overall mean F1: {df["f1"].mean():.3f}')
print(f'Overall mean F1 (non-zero): {df[df["f1"]>0]["f1"].mean():.3f}')
print()
print('By runtime:')
print(df.groupby('runtime').agg(n=('f1','count'), mean=('f1','mean'), std=('f1','std')).round(3))